In [1]:
# ============================================
# MILESTONE 2 - ML MODEL TRAINING
# ============================================

import os
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

print("Loading dataset...")

# --------------------------------------------
# 1. LOAD DATA
# --------------------------------------------

df = pd.read_csv("../datasets/Smart_Farming_Crop_Yield_2024.csv")

print("Dataset Shape:", df.shape)
display(df.head())

# --------------------------------------------
# 2. BASIC PREPROCESSING
# --------------------------------------------

target = "yield_kg_per_hectare"

# Convert dates into useful numerical features
for col in ["sowing_date", "harvest_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

if "sowing_date" in df.columns:
    df["sowing_month"] = df["sowing_date"].dt.month
    df["sowing_dayofyear"] = df["sowing_date"].dt.dayofyear

if "harvest_date" in df.columns:
    df["harvest_month"] = df["harvest_date"].dt.month
    df["harvest_dayofyear"] = df["harvest_date"].dt.dayofyear

# Drop columns that should not directly be used for prediction
drop_cols = [
    target,
    "farm_id",
    "sensor_id",
    "timestamp",
    "sowing_date",
    "harvest_date"
]

drop_cols = [c for c in drop_cols if c in df.columns]

X = df.drop(columns=drop_cols)
y = df[target]

print("\nFeatures:", X.shape[1])
print("Target:", target)

# --------------------------------------------
# 3. TRAIN / TEST SPLIT
# --------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

# --------------------------------------------
# 4. IDENTIFY FEATURES
# --------------------------------------------

numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nNumerical features:", numerical_features)
print("\nCategorical features:", categorical_features)

# --------------------------------------------
# 5. PREPROCESSING PIPELINE
# --------------------------------------------

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

# --------------------------------------------
# 6. DIFFERENT ML MODELS + GRID SEARCH
# --------------------------------------------

models = {

    "Linear Regression": (
        LinearRegression(),
        {}
    ),

    "Random Forest": (
        RandomForestRegressor(
            random_state=42,
            n_jobs=-1
        ),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [None, 10, 20],
            "model__min_samples_split": [2, 5]
        }
    ),

    "Gradient Boosting": (
        GradientBoostingRegressor(
            random_state=42
        ),
        {
            "model__n_estimators": [100, 200],
            "model__learning_rate": [0.05, 0.1],
            "model__max_depth": [2, 3]
        }
    ),

    "Extra Trees": (
        ExtraTreesRegressor(
            random_state=42,
            n_jobs=-1
        ),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [None, 10, 20],
            "model__min_samples_split": [2, 5]
        }
    )
}

results = []
trained_models = {}

# --------------------------------------------
# 7. TRAIN AND EVALUATE
# --------------------------------------------

for name, (model, params) in models.items():

    print("\n" + "=" * 60)
    print("Training:", name)
    print("=" * 60)

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    if params:
        grid = GridSearchCV(
            pipeline,
            params,
            cv=5,
            scoring="neg_mean_absolute_error",
            n_jobs=-1,
            verbose=0
        )

        grid.fit(X_train, y_train)

        best_model = grid.best_estimator_
        best_params = grid.best_params_

        print("Best Parameters:", best_params)

    else:
        pipeline.fit(X_train, y_train)
        best_model = pipeline
        best_params = {}

    predictions = best_model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    print("MAE :", round(mae, 2))
    print("RMSE:", round(rmse, 2))
    print("R2  :", round(r2, 4))

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    })

    trained_models[name] = best_model

# --------------------------------------------
# 8. MODEL COMPARISON
# --------------------------------------------

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="MAE",
    ascending=True
).reset_index(drop=True)

print("\n\nMODEL COMPARISON")
display(results_df)

# --------------------------------------------
# 9. BEST MODEL
# --------------------------------------------

best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("\nBest Model:", best_model_name)
print("Best MAE:", round(results_df.iloc[0]["MAE"], 2))
print("Best RMSE:", round(results_df.iloc[0]["RMSE"], 2))
print("Best R2 Score:", round(results_df.iloc[0]["R2 Score"], 4))

# --------------------------------------------
# 10. SAMPLE PREDICTION
# --------------------------------------------

sample = X_test.iloc[[0]]

sample_prediction = best_model.predict(sample)[0]

print("\nSample Actual Yield:",
      round(y_test.iloc[0], 2),
      "kg/hectare")

print("Sample Predicted Yield:",
      round(sample_prediction, 2),
      "kg/hectare")

# --------------------------------------------
# 11. SAVE BEST MODEL
# --------------------------------------------

os.makedirs("../models", exist_ok=True)

model_path = "../models/best_crop_yield_model.joblib"

joblib.dump(best_model, model_path)

print("\nBest model saved successfully!")
print("Location:", model_path)

# --------------------------------------------
# 12. FINAL MESSAGE
# --------------------------------------------

print("\n" + "=" * 60)
print("MILESTONE 2 MODEL TRAINING COMPLETED SUCCESSFULLY!")
print("=" * 60)

Loading dataset...
Dataset Shape: (500, 22)


,farm_id,region,crop_type,soil_moisture_%,soil_pH,temperature_C,rainfall_mm,humidity_%,sunlight_hours,irrigation_type,...,sowing_date,harvest_date,total_days,yield_kg_per_hectare,sensor_id,timestamp,latitude,longitude,NDVI_index,crop_disease_status
0,FARM0001,North India,Wheat,35.95,5.99,17.79,75.62,77.03,7.27,NaN,...,2024-01-08,2024-05-09,122,4408.07,SENS0001,2024-03-19,14.970941,82.997689,0.63,Mild
1,FARM0002,South USA,Soybean,19.74,7.24,30.18,89.91,61.13,5.67,Sprinkler,...,2024-02-04,2024-05-26,112,5389.98,SENS0002,2024-04-21,16.613022,70.869009,0.58,NaN
2,FARM0003,South USA,Wheat,29.32,7.16,27.37,265.43,68.87,8.23,Drip,...,2024-02-03,2024-06-26,144,2931.16,SENS0003,2024-02-28,19.503156,79.068206,0.80,Mild
3,FARM0004,Central USA,Maize,17.33,6.03,33.73,212.01,70.46,5.03,Sprinkler,...,2024-02-21,2024-07-04,134,4227.80,SENS0004,2024-05-14,31.071298,85.519998,0.44,NaN
4,FARM0005,Central USA,Cotton,19.37,5.92,33.86,269.09,55.73,7.93,NaN,...,2024-02-05,2024-05-20,105,4979.96,SENS0005,2024-04-13,16.568540,81.691720,0.84,Severe



Features: 20
Target: yield_kg_per_hectare

Training samples: 400
Testing samples: 100

Numerical features: ['soil_moisture_%', 'soil_pH', 'temperature_C', 'rainfall_mm', 'humidity_%', 'sunlight_hours', 'pesticide_usage_ml', 'total_days', 'latitude', 'longitude', 'NDVI_index']

Categorical features: ['region', 'crop_type', 'irrigation_type', 'fertilizer_type', 'crop_disease_status']

Training: Linear Regression
MAE : 1094.99
RMSE: 1226.3
R2  : -0.0889

Training: Random Forest
Best Parameters: {'model__max_depth': 10, 'model__min_samples_split': 2, 'model__n_estimators': 200}
MAE : 1060.34
RMSE: 1210.03
R2  : -0.0602

Training: Gradient Boosting
Best Parameters: {'model__learning_rate': 0.05, 'model__max_depth': 2, 'model__n_estimators': 100}
MAE : 1085.45
RMSE: 1232.69
R2  : -0.1003

Training: Extra Trees
Best Parameters: {'model__max_depth': 10, 'model__min_samples_split': 2, 'model__n_estimators': 100}
MAE : 1062.84
RMSE: 1213.25
R2  : -0.0658


MODEL COMPARISON


,Model,MAE,RMSE,R2 Score
0,Random Forest,1060.339764,1210.033650,-0.060186
1,Extra Trees,1062.844559,1213.248550,-0.065827
2,Gradient Boosting,1085.450943,1232.687747,-0.100255
3,Linear Regression,1094.985700,1226.302997,-0.088887



Best Model: Random Forest
Best MAE: 1060.34
Best RMSE: 1210.03
Best R2 Score: -0.0602

Sample Actual Yield: 5867.22 kg/hectare
Sample Predicted Yield: 3919.25 kg/hectare

Best model saved successfully!
Location: ../models/best_crop_yield_model.joblib

MILESTONE 2 MODEL TRAINING COMPLETED SUCCESSFULLY!


# Milestone 2 – ML Model Training and Predictive Analytics

## Objectives
1. Train different ML models using GridSearchCV.
2. Compare models using evaluation metrics.
3. Select the best-performing model.
4. Understand predictive analytics and agricultural forecasting.
5. Generate AI-based agricultural insights using an external LLM.